In [1]:
import requests
import json
from datetime import datetime
import pandas as pd

# STEP 1: Basic weather search for a city

In [2]:
def get_weather(latitude, longitude, city_name):
    """
    Gets weather data from the Open-Meteo API
    """
    url = "https://api.open-meteo.com/v1/forecast"

    params = {
        "latitude": latitude,
        "longitude": longitude,
        "current": "temperature_2m,relative_humidity_2m,weather_code",
        "hourly": "temperature_2m,precipitation",
        "daily": "weather_code,temperature_2m_max,temperature_2m_min,precipitation_sum",
        "timezone": "auto"
    }

    response = requests.get(url, params=params)

    if response.status_code == 200:
        data = response.json()
        return data
    else:
        print(f"Error: {response.status_code}")
        return None



# STEP 2: Interpreting the Weather Code

In [10]:
def interpret_weather_code(code):
    """
    Converts weather code to description
    """
    weather_codes = {
        0: "Clear sky",
        1: "Mostly clean",
        2: "Partly cloudy",
        3: "Cloudy",
        45: "Fog",
        48: "Frozen fog",
        51: "Drizzle",
        61: "Rain",
        63: "Moderate rain",
        65: "Heavy rain",
        71: "Snow",
        80: "Light rain",
        82: "Heavy rainfall",
        95: "Storm"
    }
    return weather_codes.get(code, "Unknown")

# STEP 3: Presenting the data in a nice format

In [11]:
def display_weather(city_name, data):
    """
    Displays weather data in structured format
    """
    if data is None:
        return

    current = data.get("current", {})
    daily = data.get("daily", {})

    print(f"\n{'='*50}")
    print(f"WEATHER IN {city_name.upper()}")
    print(f"{'='*50}\n")

    # Current weather
    print("CURRENT WEATHER:")
    print(f"  • Temperature: {current.get('temperature_2m')}°C")
    print(f"  • Humidity: {current.get('relative_humidity_2m')}%")
    print(f"  • Current: {interpret_weather_code(current.get('weather_code'))}\n")

    # 7-day forecast
    print("7 DAY FORECAST:")
    times = daily.get("time", [])
    temps_max = daily.get("temperature_2m_max", [])
    temps_min = daily.get("temperature_2m_min", [])
    weather_codes = daily.get("weather_code", [])

    for i in range(min(7, len(times))):
        date = times[i]
        max_temp = temps_max[i]
        min_temp = temps_min[i]
        weather = interpret_weather_code(weather_codes[i])
        print(f"  {date}: {max_temp}°C / {min_temp}°C - {weather}")


# STEP 4: Running the program

In [12]:
#Coordinates of various cities

cities = {
    "Thessaloniki": (40.6353, 22.9375),
    "Athens": (37.9838, 23.7275),
    "London": (51.5074, -0.1278),
    "New York": (40.7128, -74.0060),
}

# We get data about cities

for city_name, (lat, lon) in cities.items():
    weather_data = get_weather(lat, lon, city_name)
    display_weather(city_name, weather_data)


WEATHER IN THESSALONIKI

CURRENT WEATHER:
  • Temperature: 17.0°C
  • Humidity: 79%
  • Current: Cloudy

7 DAY FORECAST:
  2025-11-21: 19.3°C / 16.9°C - Rain
  2025-11-22: 20.5°C / 15.6°C - Light rain
  2025-11-23: 17.1°C / 9.9°C - Cloudy
  2025-11-24: 13.8°C / 6.1°C - Clear sky
  2025-11-25: 15.6°C / 7.1°C - Cloudy
  2025-11-26: 14.9°C / 7.4°C - Light rain
  2025-11-27: 16.3°C / 11.4°C - Storm

WEATHER IN ATHENS

CURRENT WEATHER:
  • Temperature: 18.3°C
  • Humidity: 79%
  • Current: Partly cloudy

7 DAY FORECAST:
  2025-11-21: 22.1°C / 18.0°C - Cloudy
  2025-11-22: 22.0°C / 18.2°C - Light rain
  2025-11-23: 18.6°C / 12.1°C - Cloudy
  2025-11-24: 16.9°C / 7.9°C - Clear sky
  2025-11-25: 17.5°C / 7.9°C - Cloudy
  2025-11-26: 19.7°C / 13.7°C - Storm
  2025-11-27: 19.7°C / 15.9°C - Cloudy

WEATHER IN LONDON

CURRENT WEATHER:
  • Temperature: 1.7°C
  • Humidity: 86%
  • Current: Mostly clean

7 DAY FORECAST:
  2025-11-21: 5.9°C / 1.1°C - Unknown
  2025-11-22: 8.4°C / 1.9°C - Light rain
 

# STEP 5: Create CSV with the data

In [13]:
def save_weather_to_csv(cities_dict, filename="weather_data.csv"):
    """
    Saves weather data to a CSV file
    """
    rows = []

    for city_name, (lat, lon) in cities_dict.items():
        weather_data = get_weather(lat, lon, city_name)

        if weather_data:
            current = weather_data.get("current", {})
            rows.append({
                "City": city_name,
                "Temperature (°C)": current.get("temperature_2m"),
                "Humidity (%)": current.get("relative_humidity_2m"),
                "Current": interpret_weather_code(current.get("weather_code")),
                "Timestamp": datetime.now().isoformat()
            })

    df = pd.DataFrame(rows)
    df.to_csv(filename, index=False, encoding="utf-8-sig")
    print(f"\n✓ Data stored in {filename}")
    print(df)

# Saving the data
save_weather_to_csv(cities)


✓ Data stored in weather_data.csv
           City  Temperature (°C)  Humidity (%)        Current  \
0  Thessaloniki              17.0            79         Cloudy   
1        Athens              18.3            79  Partly cloudy   
2        London               1.7            86   Mostly clean   
3      New York               3.1            67      Clear sky   

                    Timestamp  
0  2025-11-21T01:13:23.139638  
1  2025-11-21T01:13:23.608842  
2  2025-11-21T01:13:24.079695  
3  2025-11-21T01:13:24.546983  
